# Imports

In [1]:
from cyvcf2 import VCF
import pandas as pd
import numpy as np
from scipy import stats

# Load Data

In [2]:
vcf = VCF("../data/ps3_gwas.vcf.gz")

In [3]:
phen = pd.read_csv("../data/ps3_gwas.phen", sep="\t", header=None)

In [4]:
pcs = pd.read_csv("../python_results/eigenvec.txt", sep=r"\s+", header=None)

# Clean and Align Phenotype Data

In [5]:
phen.columns = ["FID","IID","PHEN"]
phen = phen.set_index("IID")
samples = vcf.samples
y = phen.loc[samples]["PHEN"].values

# GWAS w/out PCA

In [6]:
# Set up output file
with open("../python_results/gwas_results.tsv", "w") as out:
    out.write("CHR\tBP\tSNP\tBETA\tP\tN\n")

    for v in vcf:
        # Convert genotype to dosage-like numeric
        g = v.gt_types.astype(float)
        missing_mask = (g == 2) # UNKNOWN
        g[g == 3] = 2 # HOM_ALT -> dosage 2
        g[missing_mask] = np.nan # UNKNOWN -> NaN

        # Per-SNP complete-case mask
        mask = ~np.isnan(g) & ~np.isnan(y)
        g2 = g[mask]
        y2 = y[mask]

        # Flip allele to match PLINK using minor allele, not ALT allele as effect allele
        if g2.mean() / 2 > 0.5:
            g2 = 2.0 - g2
            
        # Skip if not enough samples
        if len(g2) < 3:
            continue

        # MAF on non-missing genotypes
        maf = min(g2.mean() / 2, 1 - g2.mean() / 2)
        if maf < 0.01:
            continue

        # Regression on filtered samples
        y0 = y2 - y2.mean()
        g0 = g2 - g2.mean()
        df = len(y2) - 2

        denom = g0 @ g0
        if denom == 0: # Skip monomorphic SNPs (only 1 allele present in all samples)
            continue

        beta = (g0 @ y0) / denom

        resid = y0 - beta * g0
        se = np.sqrt((resid @ resid) / df / denom)

        t = beta / se
        p = 2 * stats.t.sf(abs(t), df)

        # Check SNPs
        snp_id = v.ID if v.ID is not None else f"{v.CHROM}:{v.POS}"

        # Write and increment
        out.write(f"{v.CHROM}\t{v.POS}\t{snp_id}\t{beta}\t{p}\t{len(y2)}\n")

# Clean and Align PC Data

In [7]:
pcs.columns = ["FID", "IID"] + [f"PC{i}" for i in range(1, pcs.shape[1] - 1)]
pcs = pcs.set_index("IID")
pc_matrix = pcs.loc[samples].drop(columns=["FID"]).values.astype(float)

intercept = np.ones(len(pc_matrix))
C = np.column_stack([intercept, pc_matrix])

# GWAS w/PCA

In [8]:
# Reload VCF
vcf = VCF("../data/ps3_gwas.vcf.gz")

In [9]:
# Compute y-residuals
covar_coefs, *_ = np.linalg.lstsq(C, y, rcond=None)
y_resid = y - C @ covar_coefs

# Precompute to save computation later
covar_complete = ~np.any(np.isnan(C), axis=1)

In [10]:
# Set up output file
with open("../python_results/gwas_results_covar.tsv", "w") as out:
    out.write("CHR\tBP\tSNP\tBETA\tP\tN\n")

    for v in vcf:
        # Convert genotype to dosage-like numeric
        g = v.gt_types.astype(float)
        missing_mask = (g == 2) # UNKNOWN
        g[g == 3] = 2 # HOM_ALT -> dosage 2
        g[missing_mask] = np.nan # UNKNOWN -> NaN

        # Per-SNP complete-case mask
        mask = ~np.isnan(g) & ~np.isnan(y) & covar_complete
        g2 = g[mask]
        y2 = y_resid[mask]
        C2 = C[mask]

        # Flip allele to match PLINK using minor allele, not ALT allele as effect allele
        if g2.mean() / 2 > 0.5:
            g2 = 2.0 - g2

        # Skip if not enough samples
        if len(g2) < C2.shape[1] + 2:
            continue

        # Compute g-residuals
        covar_coefs_g, *_ = np.linalg.lstsq(C2, g2, rcond=None)
        g_resid = g2 - C2 @ covar_coefs_g 

        # MAF on non-missing genotypes
        maf = min(g2.mean() / 2, 1 - g2.mean() / 2)
        if maf < 0.01:
            continue

        # Regression on filtered samples
        denom = g_resid @ g_resid
        if denom == 0: # Skip monomorphic SNPs (only 1 allele present in all samples)
            continue

        beta = (g_resid @ y2) / denom

        df = len(y2) - C2.shape[1] - 1
        resid = y2 - beta * g_resid
        se = np.sqrt((resid @ resid) / df / denom)

        t = beta / se
        p = 2 * stats.t.sf(abs(t), df)

        # Check SNPs
        snp_id = v.ID if v.ID is not None else f"{v.CHROM}:{v.POS}"

        # Write and increment
        out.write(f"{v.CHROM}\t{v.POS}\t{snp_id}\t{beta}\t{p}\t{len(y2)}\n")

# Clumping

In [11]:
def clump(results_file_name, p1 = 5e-8, p2 = 1e-5, kb_window = 250, r2_threshold = 0.1):
    # Load results
    gwas = pd.read_csv(f"../python_results/{results_file_name}.tsv", sep="\t")
    gwas_filtered = gwas[gwas["P"] <= p2].copy()
    gwas_bp = gwas_filtered.sort_values("BP").reset_index(drop=True)

    # SNPs actually needed for LD
    needed_snps = set(gwas_bp["SNP"])

    # Load VCF and keep only needed SNPs
    vcf = VCF("../data/ps3_gwas.vcf.gz")
    geno_dict = {}
    for v in vcf:
        if v.ID not in needed_snps:
            continue

        g = v.gt_types.astype(float)
        missing_mask = (g == 2) # UNKNOWN
        g[g == 3] = 2 # HOM_ALT -> dosage 2
        g[missing_mask] = np.nan # UNKNOWN -> NaN

        geno_dict[v.ID] = g

    # Define function to find LD for a pari
    def ld_r2(snp1, snp2):
        g1 = geno_dict[snp1]
        g2 = geno_dict[snp2]

        # Filter out NaNs and check if at least 3 overlapping samples
        mask = ~np.isnan(g1) & ~np.isnan(g2)
        if mask.sum() < 3:
            return np.nan

        # Correlation coefficient calculation
        x = g1[mask]
        y = g2[mask]

        x_mean = x.mean()
        y_mean = y.mean()

        cov = np.sum((x - x_mean) * (y - y_mean))
        var_x = np.sum((x - x_mean) ** 2)
        var_y = np.sum((y - y_mean) ** 2)

        if var_x == 0 or var_y == 0:
            return np.nan

        r = cov / np.sqrt(var_x * var_y)

        return r * r

    # Separate p-value order for choosing lead SNPs
    order = gwas_bp.sort_values("P").index

    row_snps = []
    removed = set()
    bp_window = kb_window * 1000

    for idx in order:
        row = gwas_bp.loc[idx]
        row_snp = row["SNP"]

        # Check if already clumped
        if row_snp in removed:
            continue

        # Only SNPs passing p1 can start a clump
        if row["P"] > p1:
            continue

        # Add as lead SNP
        row_snps.append(row_snp)
        row_bp = row["BP"]

        # Scan left side of window
        j = idx - 1
        while j >= 0:
            other = gwas_bp.loc[j]
            other_snp = other["SNP"]

            dist_bp = row_bp - other["BP"]
            if dist_bp > bp_window:
                break

            if other_snp not in removed:
                r2 = ld_r2(row_snp, other_snp)
                if not np.isnan(r2) and r2 >= r2_threshold:
                    removed.add(other_snp)

            j -= 1

        # Scan right side of window
        j = idx + 1
        while j < len(gwas_bp):
            other = gwas_bp.loc[j]
            other_snp = other["SNP"]

            dist_bp = other["BP"] - row_bp
            if dist_bp > bp_window:
                break

            if other_snp not in removed:
                r2 = ld_r2(row_snp, other_snp)
                if not np.isnan(r2) and r2 >= r2_threshold:
                    removed.add(other_snp)

            j += 1

    clumped = gwas_bp[gwas_bp["SNP"].isin(row_snps)].copy()
    clumped.to_csv(f"../python_results/{results_file_name}_clumped.tsv", sep="\t", index=False)

In [12]:
clump('gwas_results')

In [13]:
clump('gwas_results_covar')